[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TunaLee/posco/blob/main/notebooks/day13_solution.ipynb)

# Day 13 · 정답 — MCP 서버 만들기 · 앱에 붙이기

노트북 안에 있던 서버를 파일로 떼어 내고, 띄우고, 다른 앱이 붙게 한다

---

### 시작하기 전에

1. **파일 → 드라이브에 사본 저장** 을 먼저 누른다. 안 하면 고친 내용이 남지 않는다.
2. 셀을 고르고 **Shift + Enter** 로 실행한다.

`live` 와 `lab` 의 모든 문제에 대한 정답본이다.
수강생은 먼저 스스로 풀어 본 뒤에 연다.

두 벌을 합쳐 담으므로 **문제 번호가 `lab` 과 다르다.** 번호 대신
**지문으로 찾는다.**

문제는 실행하면 `assert` 로 자가 채점된다. 맞으면 `통과` 가 찍히고,
틀리면 기대값과 실제값이 같이 나온다.

## 1. 서버를 파일로 떼어 내기

In [ ]:
# FastMCP 를 받는다
!pip install -q fastmcp

import json, socket, threading, time, urllib.request
import pandas as pd
from fastmcp import Client, FastMCP

df = pd.read_csv('https://tunalee.github.io/posco/data/cell_process.csv')
print('%d행 %d열' % df.shape)

In [ ]:
# 파일 하나가 곧 서버다
SERVER = '''
import pandas as pd
from fastmcp import FastMCP

df = pd.read_csv("https://tunalee.github.io/posco/data/cell_process.csv")
mcp = FastMCP("공정 도우미")

OPEN = ["로트번호", "시각", "설비호기", "교대조", "판정"]   # 내보내도 되는 칸
LIMIT = 20                                                # 한 번에 주는 최대 행수

@mcp.tool()
def defect_rate(machine: str, shift: str = "") -> str:
    "설비호기의 불량률을 돌려준다. 교대조를 주면 그 안에서만 센다."
    d = df[df["설비호기"] == machine]
    if shift:
        d = d[d["교대조"] == shift]
    if not len(d):
        return "해당 조건에 데이터가 없다"
    bad = int((d["판정"] == "불량").sum())
    return "%s %s 측정 %d건 중 불량 %d건 · %.1f%%" % (
        machine, shift or "전체", len(d), bad, 100.0 * bad / len(d))

@mcp.tool()
def recent_lots(machine: str, n: int = 5) -> str:
    "설비의 최근 로트 기록을 돌려준다. 공정 조건값은 내보내지 않는다."
    d = df[df["설비호기"] == machine].tail(min(n, LIMIT))
    return d[OPEN].to_string(index=False) if len(d) else "해당 설비가 없다"

if __name__ == "__main__":
    mcp.run()
'''
open('server.py', 'w', encoding='utf-8').write(SERVER)
print(open('server.py', encoding='utf-8').read()[:300])

## 2. 띄우고 주소 얻기

In [ ]:
# 같은 서버를 이번엔 HTTP 로 띄운다. 딴 갈래로 돌려야 셀이 안 막힌다.
srv = FastMCP('공정 도우미')

@srv.tool()
def defect_rate(machine: str, shift: str = '') -> str:
    '''설비호기의 불량률을 돌려준다. 교대조를 주면 그 안에서만 센다.

    machine: 설비호기. 1호기 ~ 4호기
    shift: 교대조. 주간 또는 야간. 비우면 전체
    '''
    d = df[df['설비호기'] == machine]
    if shift:
        d = d[d['교대조'] == shift]
    if not len(d):
        return '해당 조건에 데이터가 없다'
    bad = int((d['판정'] == '불량').sum())
    return '%s %s 측정 %d건 중 불량 %d건 · %.1f%%' % (
        machine, shift or '전체', len(d), bad, 100.0 * bad / len(d))

PORT = 8931
threading.Thread(target=lambda: srv.run(transport='http', host='127.0.0.1', port=PORT),
                 daemon=True).start()

for _ in range(60):                       # 포트가 열릴 때까지 기다린다
    s = socket.socket(); s.settimeout(0.3)
    if s.connect_ex(('127.0.0.1', PORT)) == 0:
        break
    time.sleep(0.3)

ADDR = 'http://127.0.0.1:%d/mcp' % PORT
print('서버가 떴다 —', ADDR)

## 3. 주소로 붙기

In [ ]:
# 괄호 안만 바뀐다. 그 뒤는 어제와 한 글자도 안 다르다.
async def peek(target):
    async with Client(target) as c:
        names = [t.name for t in await c.list_tools()]
        out = (await c.call_tool('defect_rate', {'machine': '3호기'})).content[0].text
        return names, out

print('객체로 붙기 :', await peek(srv))
print('주소로 붙기 :', await peek(ADDR))

### 같이 풀기

수업 중에 같이 푼다.

> **실습문제 1.** 포트를 **8932** 로 바꿔 서버를 하나 더 띄우고, 주소로 붙어 본다.
> 같은 기계에 서버 둘이 동시에 돈다. 포트가 이름표다.

In [ ]:
srv2 = FastMCP('둘째 서버')

@srv2.tool()
def machine_list() -> str:
    '''쓸 수 있는 설비호기 이름을 모두 돌려준다'''
    return ', '.join(sorted(df['설비호기'].unique()))

PORT2 = 8932
threading.Thread(target=lambda: srv2.run(transport='http', host='127.0.0.1', port=PORT2),
                 daemon=True).start()

for _ in range(60):
    s = socket.socket(); s.settimeout(0.3)
    if s.connect_ex(('127.0.0.1', PORT2)) == 0: break
    time.sleep(0.3)
async with Client('http://127.0.0.1:%d/mcp' % PORT2) as c:
    print('붙은 도구:', [t.name for t in await c.list_tools()])

## 4. 모델에게 넘어가는 것

In [ ]:
# 어제 쓰던 키를 그대로 쓴다
import getpass
KEY = getpass.getpass('nvapi- 로 시작하는 키: ')

URL = 'https://integrate.api.nvidia.com/v1/chat/completions'
MODEL = 'nvidia/llama-3.3-nemotron-super-49b-v1'

def chat(messages, tools=None, n=600):
    body = {'model': MODEL, 'max_tokens': n, 'temperature': 0, 'messages': messages}
    if tools:
        body['tools'] = tools
    req = urllib.request.Request(URL, data=json.dumps(body).encode(), headers={
        'Authorization': 'Bearer ' + KEY,
        'Content-Type': 'application/json', 'Accept': 'application/json'})
    with urllib.request.urlopen(req, timeout=180) as f:
        return json.load(f)['choices'][0]['message']

In [ ]:
# 어제 만든 고리 그대로. 클라이언트가 목록을 받고, 모델이 고른다.
def to_openai(tools):
    return [{'type': 'function',
             'function': {'name': t.name, 'description': t.description,
                          'parameters': t.inputSchema}}
            for t in tools]

async def run(target, question, log=True):
    called = []
    async with Client(target) as client:
        spec = to_openai(await client.list_tools())
        messages = [{'role': 'system', 'content':
                     '너는 공정 데이터를 보는 비서다. 한국어로만 답한다. '
                     '숫자는 도구로 조회한 값만 쓴다.'},
                    {'role': 'user', 'content': question}]
        for _ in range(4):
            m = chat(messages, spec)
            messages.append(m)
            calls = m.get('tool_calls') or []
            if not calls:
                return (m.get('content') or '').strip() or '[도구를 안 불렀다]', called
            for c in calls:
                name = c['function']['name']
                args = json.loads(c['function']['arguments'] or '{}')
                called.append(name)
                if log:
                    print('  [MCP] %s(%s)' % (name, args))
                try:
                    out = (await client.call_tool(name, args)).content[0].text
                except Exception as e:          # 터져도 그 말을 모델에게 넘긴다
                    out = '도구 실행 실패: %s' % e
                messages.append({'role': 'tool', 'tool_call_id': c['id'], 'content': out})
    return '[한도]', called

## 5. 도구 고르기가 흔들리는 것

In [ ]:
# 이름이 비슷한 도구 둘을 한 서버에 둔다
pick = FastMCP('고르기 시험')

@pick.tool()
def check_quality(machine: str) -> str:
    '''설비의 불량률을 돌려준다. 품질·불량·양품을 물으면 이것을 쓴다.

    machine: 설비호기. 1호기 ~ 4호기
    '''
    d = df[df['설비호기'] == machine]
    bad = int((d['판정'] == '불량').sum())
    return '%s 측정 %d건 중 불량 %d건 · %.1f%%' % (machine, len(d), bad,
                                              100.0 * bad / len(d))

@pick.tool()
def check_runtime(machine: str) -> str:
    '''설비가 얼마나 돌았는지 가동시간을 돌려준다. 가동·시간을 물으면 이것을 쓴다.

    machine: 설비호기. 1호기 ~ 4호기
    '''
    return '%s 가동 %d분' % (machine, len(df[df['설비호기'] == machine]) * 3)

print('도구 둘을 붙였다')

### 스스로 풀기

각자 푼다. 막히면 손을 든다.

> **빈칸 문제 1.** 도구 이름을 **분명하게** 바꿔 본다. `lookup` 은 무엇을 하는지 안 보인다.
> 이름만으로 무엇을 돌려주는지 알 수 있게 짓는다.

In [ ]:
vague = FastMCP('이름이 흐린 서버')

@vague.tool()
def compare_shifts(machine: str) -> str:
    '''한 설비의 주간조와 야간조 불량률을 나란히 견준다.
    어느 교대조가 더 나쁜지 물으면 이것을 쓴다.

    machine: 설비호기. 1호기 ~ 4호기
    '''
    d = df[df['설비호기'] == machine]
    return ' · '.join('%s %.1f%%' % (s, 100.0 * (g['판정'] == '불량').mean())
                      for s, g in d.groupby('교대조'))

answer, called = await run(vague, '3호기는 주간과 야간 중 어느 쪽이 불량이 많나', log=False)
print('부른 도구:', called or '없음')
print(answer[:150])

## 6. 나쁜 입력 막기

In [ ]:
# 검증이 없는 도구와, 길을 알려 주는 도구
rude = FastMCP('막 만든 서버')
@rude.tool()
def defect_rate(machine: str) -> str:
    '''설비호기의 불량률을 돌려준다'''
    d = df[df['설비호기'] == machine]
    bad = int((d['판정'] == '불량').sum())
    return '%.1f%%' % (100.0 * bad / len(d))        # 없는 설비면 0 으로 나눈다

kind = FastMCP('친절한 서버')
@kind.tool()
def defect_rate(machine: str) -> str:
    '''설비호기의 불량률을 돌려준다. 설비 품질을 물으면 이것을 쓴다.

    machine: 설비호기. 1호기 ~ 4호기
    '''
    ok = sorted(df['설비호기'].unique())
    if machine not in ok:
        return '「%s」 는 없는 설비다. 쓸 수 있는 이름: %s' % (machine, ', '.join(ok))
    d = df[df['설비호기'] == machine]
    bad = int((d['판정'] == '불량').sum())
    return '%s 측정 %d건 중 불량 %d건 · %.1f%%' % (machine, len(d), bad,
                                              100.0 * bad / len(d))

print('준비됐다')

## 7. 열 것과 안 열 것

In [ ]:
# 읽기 도구와 쓰기 도구를 표시로 갈라 둔다
memo = FastMCP('점검 메모')
NOTES = []

@memo.tool(annotations={'readOnlyHint': True})
def list_notes() -> str:
    '''적어 둔 점검 메모를 모두 돌려준다'''
    return '\n'.join('%d. %s' % (i + 1, n) for i, n in enumerate(NOTES)) or '메모가 없다'

@memo.tool(annotations={'readOnlyHint': False, 'destructiveHint': False})
def add_note(text: str) -> str:
    '''점검 메모를 새로 적는다. 기존 메모는 건드리지 않는다.

    text: 적을 내용
    '''
    NOTES.append(text)
    return '적었다. 지금 메모 %d개' % len(NOTES)

async with Client(memo) as c:
    for t in await c.list_tools():
        ann = t.annotations
        print('%-12s 읽기전용 %s' % (t.name, getattr(ann, 'readOnlyHint', None) if ann else None))

### 스스로 풀기

각자 푼다. 막히면 손을 든다.

> **빈칸 문제 2.** **읽기 전용 경계**를 도구 안에 넣는다. 공정 조건값은 절대 안 나가게 한다.
> `OPEN` 에 적힌 칸만 내보낸다. 나머지는 이름조차 안 보인다.

In [ ]:
safe = FastMCP('경계 있는 서버')
OPEN = ['로트번호', '시각', '설비호기', '교대조', '판정']

@safe.tool()
def recent_lots(machine: str, n: int = 5) -> str:
    '''설비의 최근 로트 기록을 돌려준다. 공정 조건값은 내보내지 않는다.

    machine: 설비호기. 1호기 ~ 4호기
    n: 몇 건. 최대 20
    '''
    d = df[df['설비호기'] == machine].tail(min(n, 20))
    return d[OPEN].to_string(index=False) if len(d) else '해당 설비가 없다'

SHUT = [c for c in df.columns if c not in OPEN]
async with Client(safe) as c:
    out = (await c.call_tool('recent_lots', {'machine': '3호기', 'n': 100})).content[0].text
print('막은 칸이 샜나:', [x for x in SHUT if x in out] or '안 샜다')
print('돌려준 줄 수:', out.count(chr(10)))

## 8. 서버 여럿 붙이기

In [ ]:
# 두 조가 각각 서버를 냈다. 그런데 도구 이름이 겹친다.
team1 = FastMCP('1조 설비')
@team1.tool()
def defect_rate(machine: str) -> str:
    '''설비호기의 불량률을 돌려준다'''
    return '[1조] %s 12.0%%' % machine

team2 = FastMCP('2조 품질')
@team2.tool()
def find_rule(question: str) -> str:
    '''규정을 찾는다'''
    return '[2조] 제26조(작업중지 등)'
@team2.tool()
def defect_rate(machine: str) -> str:       # 1조와 이름이 같다
    '''설비호기의 불량률을 돌려준다'''
    return '[2조] %s 99.9%%' % machine

print('두 조 다 defect_rate 를 만들었다')

## 9. 컨테이너로 싸기

In [ ]:
# 서버가 쓰는 꾸러미를 적어 둔다
open('requirements.txt', 'w').write('fastmcp==3.4.7\npandas\n')

DOCKERFILE = '''
FROM python:3.12-slim

WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY server.py .
EXPOSE 8000
CMD ["fastmcp", "run", "server.py",
     "--transport", "http", "--host", "0.0.0.0", "--port", "8000"]
'''
open('Dockerfile', 'w').write(DOCKERFILE.strip() + '\n')
print(open('Dockerfile').read())

## 10. compose 와 사내망

In [ ]:
# 호스트 하나에 조별 서버 둘을 붙이는 구성
COMPOSE = '''
services:
  team1:
    build: ./teams/team1
    expose: ["8000"]

  team2:
    build: ./teams/team2
    expose: ["8000"]

  host:
    build: ./host
    ports: ["8080:8080"]
    environment:
      NVIDIA_API_KEY: ${NVIDIA_API_KEY}
      TEAM_URLS: "http://team1:8000/mcp,http://team2:8000/mcp"
    depends_on: [team1, team2]
'''
open('compose.yml', 'w').write(COMPOSE.strip() + '\n')
print(open('compose.yml').read())

In [ ]:
# ① 이미지를 사내 저장소에서 받는다  ② 꾸러미도 사내 미러에서 받는다
INTRA = '''
FROM harbor.사내주소/library/python:3.12-slim

WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir \\
      --index-url https://nexus.사내주소/repository/pypi/simple \\
      --trusted-host nexus.사내주소 \\
      -r requirements.txt

COPY server.py .
CMD ["python", "server.py"]
'''
open('Dockerfile.intra', 'w').write(INTRA.strip() + '\n')
print(open('Dockerfile.intra').read())

### 스스로 풀기

각자 푼다. 막히면 손을 든다.

> **빈칸 문제 3.** compose 에 **3조 서버**를 한 칸 더 넣는다.
> 호스트가 3조를 찾아가게 `TEAM_URLS` 도 같이 고친다.

In [ ]:
import re
C = open('compose.yml').read()
C = C.replace('''  host:''', '''  team3:
    build: ./teams/team3
    expose: ["8000"]

  host:''')
C = C.replace('http://team2:8000/mcp"', 'http://team2:8000/mcp,http://team3:8000/mcp"')
C = C.replace('depends_on: [team1, team2]', 'depends_on: [team1, team2, team3]')

print(C)
assert '___' not in C, '빈 칸이 남았다'
assert C.count('build:') == 4, '서비스가 넷이어야 한다'
print('3조까지 붙는 구성이 됐다')

## 11. 조별 서버 설계

### 조별로 풀기

2~3명이 한 조로 상의하며 푼다.

> **실습문제 2.** **조별 서버**를 설계한다. 코드는 안 쓴다. 다섯 칸만 채운다.
> 2~3명이 한 조로 상의한다. 지금 손으로 하고 있는 일에서 고른다.
> 정답을 눈으로 확인할 수 있는 것을 고른다. 그래야 잘 됐는지 안다.

In [ ]:
OUR = {
    '이름':        '점검 도우미',
    '지금 손으로':  '점검 때마다 작업표준을 폴더에서 찾고 이전 기록을 엑셀에서 본다',
    '읽기 도구':    ['최근 점검 이력 조회', '작업표준 검색'],
    '쓰기 도구':    ['점검 메모 남기기'],
    '안 여는 것':   '설비 원시 계측값과 담당자 이름',
}
for k, v in OUR.items():
    print('%-10s %s' % (k, v if isinstance(v, str) else ' / '.join(v) or '없음'))
assert '___' not in str(OUR), '다섯 칸을 채운다'
print()
print('이 다섯 칸을 그대로 Codex 에 넘기면 server.py 가 나온다.')
print('「안 여는 것」이 제일 중요하다 — 그게 권한 경계다.')